# 6 PyTorch Lightning (optional)

PyTorch Lightning organizes the same model, loss and data into a module and a trainer. This notebook repeats the breast-cancer example using Lightning. If needed, install it with `%pip install lightning==2.6.6`.

In [1]:
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(2)

seed_everything(42)

## 6.1 What changes?
Lightning calls your training and validation methods and manages updates. This removes repeated loop code but does not change the learning problem.

Read the methods and identify where prediction, loss calculation and optimizer configuration moved. Lightning performs backward propagation and optimizer steps automatically under this default configuration.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def cancer_splits(seed=42):
    """60/20/20 stratified split; fit preprocessing on training data only.

    Relabel explicitly: 1 = malignant, 0 = benign (opposite sklearn's coding).
    The returned test arrays should be opened only after model selection.
    """
    source = load_breast_cancer()
    x = source.data.astype(np.float32)
    y = (source.target == 0).astype(np.float32)
    idx = np.arange(len(y))
    train, remainder = train_test_split(idx, test_size=0.4, stratify=y, random_state=seed)
    val, test = train_test_split(remainder, test_size=0.5, stratify=y[remainder], random_state=seed)
    scaler = StandardScaler().fit(x[train])
    result = {"scaler": scaler, "feature_names": source.feature_names}
    for name, rows in [("train", train), ("val", val), ("test", test)]:
        result[name] = (scaler.transform(x[rows]).astype(np.float32), y[rows])
        result[name + "_ids"] = rows
    return result

In [3]:
import lightning as L
import torch
from torch import nn
splits = cancer_splits()
x_train, y_train = splits["train"]
x_val, y_val = splits["val"]

class ClassifierModule(L.LightningModule):
    def __init__(self, lr=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(nn.Linear(30, 16), nn.ReLU(), nn.Linear(16, 1))
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, x):
        return self.model(x).squeeze(-1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, batch_size=len(y))
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("val_loss", loss, on_step=False, on_epoch=True, batch_size=len(y))

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

## 6.2 Train and retain the best validation checkpoint
We keep local checkpoints and disable external logging. A new temporary directory for each run avoids accidentally reusing a previous run's files. It is created inside `runs/` in the course folder.

The test split remains unopened in this optional organizational demonstration.

In [4]:
from torch.utils.data import DataLoader, TensorDataset

def loader(x, y, batch_size=32, shuffle=False, seed=42):
    return DataLoader(
        TensorDataset(torch.as_tensor(x, dtype=torch.float32), torch.as_tensor(y, dtype=torch.float32)),
        batch_size=batch_size, shuffle=shuffle,
        generator=torch.Generator().manual_seed(seed), num_workers=0,
    )

In [5]:
from pathlib import Path
import tempfile
from lightning.pytorch.callbacks import ModelCheckpoint
L.seed_everything(42, workers=True)
Path("runs").mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="lightning-", dir=Path("runs")))
checkpoint = ModelCheckpoint(dirpath=run_dir, monitor="val_loss", mode="min", save_top_k=1)
trainer = L.Trainer(max_epochs=10, accelerator="cpu", devices=1, logger=False,
                    callbacks=[checkpoint], enable_progress_bar=False, enable_model_summary=False,
                    default_root_dir=run_dir)
trainer.fit(ClassifierModule(), loader(x_train, y_train, shuffle=True), loader(x_val, y_val))
best = ClassifierModule.load_from_checkpoint(checkpoint.best_model_path, map_location="cpu", weights_only=True)
print("Best validation loss:", checkpoint.best_model_score.item())
print("Restored output shape:", best(torch.tensor(x_val[:1])).shape)

Best validation loss: 0.043476689606904984
Restored output shape: torch.Size([1])


*Assignment: Map the loop*

Where are `zero_grad`, `backward` and `step` now? Why must the output still be a logit? Why restore the best checkpoint rather than assuming the final epoch is best?

*Answer:*

Lightning performs the update operations automatically by default. BCEWithLogitsLoss still expects raw logits and includes sigmoid internally. Later epochs can overfit, so restore the checkpoint selected using validation. Switching libraries does not change the test-set rules.

**Optional extension:** Add a local CSV logger. For sweeps, explicitly pass each run's sampled configuration into the model constructor; merely declaring a search space does not change the model. Keep external service setup optional.